# Determinants and Inverses

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 01.04 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/05_determinants_and_inverses.ipynb)

---

## 🎯 Learning Objective

Understand what the determinant measures geometrically, when a matrix can be inverted, and how the Moore-Penrose pseudoinverse generalizes inversion to non-square matrices.

---

## 📐 Theory

### The determinant

For a $2\times2$ matrix, $\det\begin{pmatrix}a&b\\c&d\end{pmatrix} = ad-bc$. Geometrically,
$|\det(A)|$ is the factor by which $A$ scales area (in 2D) or volume (in higher dimensions);
the *sign* tells you whether orientation is preserved (positive) or flipped, like a reflection
(negative). Key properties:

$$\det(AB) = \det(A)\det(B) \qquad \det(A^\top) = \det(A) \qquad \det(A) = 0 \iff A \text{ is singular}$$

A matrix is **singular** (non-invertible) exactly when its determinant is zero — geometrically,
it collapses space into a lower dimension, and no transformation can undo that information loss.

### The inverse

$A^{-1}$ is the matrix satisfying $AA^{-1} = A^{-1}A = I$. For $2\times2$:

$$A^{-1} = \frac{1}{\det(A)}\begin{pmatrix}d & -b \\ -c & a\end{pmatrix}$$

Notice the formula divides by $\det(A)$ — this is *why* a zero determinant means no inverse
exists, and why a determinant *close* to zero causes numerical trouble (dividing by a tiny
number amplifies errors).

### The Moore-Penrose pseudoinverse

Non-square or singular matrices don't have a true inverse, but the **pseudoinverse** $A^+$
generalizes the idea: it gives the least-squares "best" solution to $A\mathbf{x}=\mathbf{b}$ for
overdetermined systems, and the minimum-norm solution for underdetermined ones. It's computed
via the SVD (which we'll properly meet in [Notebook 01.07](07_SVD_and_PCA.ipynb)):
$A^+ = V\Sigma^+U^\top$.

---

## 👁️ Visual Intuition

The determinant is literally the area-scaling factor of a transformation: watch the unit
square's area change (and its orientation flip, for a reflection) as we apply different matrices.

In [ ]:
# |det(A)| = area scaling factor; sign of det(A) = orientation preserved (+) or flipped (-)
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

unit_square = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]])  # closed polygon for plotting

matrices = {
    "Stretch (det=3)": np.array([[3.0, 0.0], [0.0, 1.0]]),
    "Shear (det=1, area preserved)": np.array([[1.0, 0.8], [0.0, 1.0]]),
    "Reflection (det=-1)": np.array([[0.0, 1.0], [1.0, 0.0]]),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (title, A) in zip(axes, matrices.items()):
    transformed = A @ unit_square
    ax.plot(*unit_square, color="gray", ls="--", label="unit square")
    ax.fill(*transformed, alpha=0.4, color="#4C72B0")
    ax.plot(*transformed, color="#4C72B0", label="transformed")
    ax.set_title(f"{title}\ndet={np.linalg.det(A):.2f}")
    ax.set_aspect('equal'); ax.set_xlim(-1.5, 3.5); ax.set_ylim(-1.5, 3.5)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Cofactor expansion for $2\times2$/$3\times3$ determinants, the $2\times2$ inverse formula, and
the pseudoinverse for a non-square matrix — each checked against NumPy.

In [ ]:
# Determinant by cofactor expansion, 2x2 inverse formula, and the pseudoinverse
import numpy as np

def det_2x2(A):
    return A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]

def det_3x3(A):
    # Cofactor expansion along the first row
    return (A[0, 0] * det_2x2(A[1:, [1, 2]])
            - A[0, 1] * det_2x2(A[1:, [0, 2]])
            + A[0, 2] * det_2x2(A[1:, [0, 1]]))

def inverse_2x2(A):
    d = det_2x2(A)
    if abs(d) < 1e-12:
        raise ValueError("matrix is singular (det ~ 0), cannot invert")
    return (1.0 / d) * np.array([[A[1, 1], -A[0, 1]], [-A[1, 0], A[0, 0]]])

A2 = np.array([[4.0, 7.0], [2.0, 6.0]])
A3 = np.array([[6.0, 1.0, 1.0], [4.0, -2.0, 5.0], [2.0, 8.0, 7.0]])

print("det_2x2 vs np.linalg.det:", det_2x2(A2), "vs", round(float(np.linalg.det(A2)), 6))
print("det_3x3 vs np.linalg.det:", det_3x3(A3), "vs", round(float(np.linalg.det(A3)), 6))
print("inverse_2x2 matches np.linalg.inv:", np.allclose(inverse_2x2(A2), np.linalg.inv(A2)))

print("\n--- Pseudoinverse for a non-square (overdetermined) matrix ---")
A_rect = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])   # 3x2, no true inverse exists
A_pinv = np.linalg.pinv(A_rect)
print("A_rect shape:", A_rect.shape, " A_pinv shape:", A_pinv.shape)
print("A_pinv @ A_rect (should be close to I_2):\n", np.round(A_pinv @ A_rect, 6))

## 🤖 Why This Matters for AI

**Near-singular matrices cause numerical instability.** When $\det(A)$ is close to zero, the
formula for $A^{-1}$ divides by a tiny number, amplifying any noise in the input into a huge
error in the output. This is directly relevant to **normalizing flows**, a family of generative
models that must use *invertible* transformations by construction — flow architectures are
carefully designed (e.g. using triangular Jacobians) specifically so their determinants stay
well away from zero and cheap to compute.

In [ ]:
# Near-singular matrices amplify tiny input errors into huge output errors
import numpy as np

def relative_error(x_true, x_approx):
    return np.linalg.norm(x_true - x_approx) / np.linalg.norm(x_true)

# A matrix that is ALMOST singular: its two rows are nearly parallel
A_nearly_singular = np.array([[1.0, 1.0], [1.0, 1.0001]])
print("det(A) =", np.linalg.det(A_nearly_singular), " (very close to zero)")

b = np.array([2.0, 2.0001])
x_true = np.linalg.solve(A_nearly_singular, b)

b_perturbed = b + np.array([1e-4, 0.0])       # a TINY perturbation to b
x_perturbed = np.linalg.solve(A_nearly_singular, b_perturbed)

print("\nOriginal solution:  ", np.round(x_true, 4))
print("Perturbed solution: ", np.round(x_perturbed, 4))
print(f"Input perturbation size:  {np.linalg.norm(b_perturbed - b):.6f}")
print(f"Output change size:       {np.linalg.norm(x_perturbed - x_true):.6f}")
print("\nA tiny nudge to b caused a huge swing in x -- this sensitivity is exactly what the")
print("CONDITION NUMBER (Notebook 01.06) will let us quantify precisely.")

## 🏋️ Exercises

### Warm-up
1. Compute $\det\begin{pmatrix}3&8\\4&6\end{pmatrix}$ and its inverse by hand, then verify with `det_2x2`/`inverse_2x2`.

### Practice
2. For two random $3\times3$ matrices $A,B$, verify numerically that $\det(AB) = \det(A)\det(B)$ and that $\det(A^\top)=\det(A)$.

### Challenge
3. Build a $3\times3$ matrix that is exactly singular (make one row a combination of the other two), confirm `np.linalg.det` gives (approximately) zero, and show that `np.linalg.inv` raises an error while `np.linalg.pinv` still returns a result. Explain what problem the pseudoinverse's result actually solves in this singular case.

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 5 (Determinants)
- Papamakarios et al., ["Normalizing Flows for Probabilistic Modeling and Inference"](https://arxiv.org/abs/1912.02762)

---

*Next notebook: [Eigenvalues and Eigenvectors](06_eigenvalues_eigenvectors.ipynb)*